# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadTalha-pk/ML-Internship-Task1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Lane: Freestyle — Growth / Recovery / Momentum Prediction  
Primary ML task type: Binary classification with a ranking/scoring output.  
The model will estimate whether a content page is at risk of meaningful decline during a future 30-day period using signals observed during an earlier 90-day period. Because the practical decision is which pages should be reviewed first, the predicted probabilities will also be used to rank pages from highest to lowest risk.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target: future_decline_30d
1 = the page experiences meaningful decline during the future 30-day outcome window.
0 = the page does not experience meaningful decline during that outcome window.
The final target will be calculated from observed future performance in the warehouse data. The exact decline threshold will be defined later in the data-contract stage. Features will come only from the earlier feature window so that future information cannot leak into training.
For this assignment, the 30,000-row starter dataset does not contain the final future 30-day outcome. Therefore, I will create a temporary proxy called starter_decline_proxy from the current trend_direction == "down" field only to demonstrate what a binary target column looks like. This proxy is only for task framing and will not be treated as the final capstone label. trend_direction and trend_pct will not be used as predictive features for this proxy.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Primary success metric: Precision@50
The practical output is a ranked list of pages for human review. If a reviewer can inspect approximately 50 pages, Precision@50 measures how many of the 50 highest-risk recommendations actually match the observed outcome.
I will also consider average precision, precision, recall, and ROC-AUC as supporting metrics, but the main business-facing metric will be Precision@50 because it matches the review decision.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: One row represents one pseudonymized content page. In this starter dataset, each content item appears once with its search, content, and engagement signals. For the final project, each training example will represent one page at a defined prediction date, using the previous 90 days of information to predict whether the page declines in the following 30 days.

In [3]:
import os
import subprocess
import pandas as pd

REPO_DIR = "/content/flyrank-ml-internship-starter"

# Clone repo if Colab does not have it yet
if not os.path.exists(REPO_DIR):
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/flyrank-bih/flyrank-ml-internship-starter",
            REPO_DIR
        ],
        check=True
    )

# Load dataset directly
file_path = f"{REPO_DIR}/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(file_path)

unit_df = df[
    [
        "content_id",
        "client_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "sessions_90d",
        "engagement_rate",
        "trend_direction"
    ]
].copy()

unit_df["starter_decline_proxy"] = (
    unit_df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

print("Rows:", len(unit_df))
print("Unique content items:", unit_df["content_id"].nunique())
print(
    "Proxy positive rate:",
    round(unit_df["starter_decline_proxy"].mean(), 3)
)

unit_df.head(10)

Rows: 30000
Unique content items: 30000
Proxy positive rate: 0.542


,content_id,client_id,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,sessions_90d,engagement_rate,trend_direction,starter_decline_proxy
0,content_304f48230142,client_f369cb89fc,3803,29,0.76,10.6,187,17,5.88,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,7,0.05,20.3,445,9,0.00,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,0.09,36.5,141,11,0.00,down,1
3,content_331d6c4de07b,client_19581e27de,11751,58,0.49,6.2,463,78,1.28,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,24,0.13,44.0,263,145,0.00,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,1,0.03,8.5,147,5,0.00,down,1
6,content_9a34b442b552,client_8722616204,20,0,0.00,7.0,90,1,0.00,down,1
7,content_a63219c6e95a,client_19581e27de,1724,1,0.06,21.2,445,28,3.57,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,29,0.09,46.0,90,68,5.88,down,1
9,content_c27558df2b0c,client_19581e27de,1240,2,0.16,4.9,257,3,0.00,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule may use only one or two conditions, such as low CTR or old content. Future decline may depend on several signals working together, including impressions, position, CTR, content age, sessions, and engagement. Machine learning can test these combinations and rank pages by risk. However, the ML model should still be compared with a simple baseline rule to check whether it actually performs better.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.